##Recuperacion de la Informacion
###Examen de 1 Bimestre
Joel Quilumba

Instalación de dependencias

In [13]:
# Instalación de librerías
!pip install -q sentence-transformers pandas numpy scikit-learn
!pip install -q sentence-transformers pandas numpy scikit-learn gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 54.5 MB/s eta 0:00:00


Importación de librerías y carga del corpus

In [3]:
import pandas as pd
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Rutas de los archivos subidos a Colab
file_movies = '/content/sample_data/rotten_tomatoes_movies.csv'
file_reviews = '/content/sample_data/rotten_tomatoes_critic_reviews.csv'

try:
    # 1. Cargar corpus
    df_movies = pd.read_csv(file_movies)
    df_reviews = pd.read_csv(file_reviews)
    print(f"Películas cargadas: {df_movies.shape[0]} registros.")
    print(f"Reseñas cargadas: {df_reviews.shape[0]} registros.")

    # 2. Unir los datasets usando el enlace como llave primaria
    df = pd.merge(
        df_reviews,
        df_movies[['rotten_tomatoes_link', 'movie_title']],
        on='rotten_tomatoes_link',
        how='inner'
    )
    print(f"Dataset unificado: {df.shape[0]} registros.")

    # 3. Filtrar columnas útiles y eliminar filas con datos nulos
    col_titulo = 'movie_title'
    col_resena = 'review_content'
    df = df[[col_titulo, col_resena]].dropna().copy()

    # 4. Concatenar título y reseña para dar mayor contexto al embedding
    df['texto_completo'] = df[col_titulo] + ". " + df[col_resena]
    print(f"Datos listos para procesar: {df.shape[0]} documentos finales.")

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Asegúrate de que ambos archivos estén correctamente subidos a la carpeta /content/sample_data/ con los nombres exactos.")

/tmp/ipykernel_4227/3385535452.py:14: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv(file_reviews)


Películas cargadas: 17712 registros.
Reseñas cargadas: 282222 registros.
Dataset unificado: 282221 registros.
Datos listos para procesar: 261561 documentos finales.


Preprocesamiento de texto

In [4]:
import unicodedata

def preprocess_text(text):
    # 0. Asegurar que sea string
    text = str(text)

    # 1. Normalización de caracteres
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

    # 2. Conversión a minúsculas
    text = text.lower()

    # 3. Eliminación de signos de puntuación
    text = re.sub(r'[^\w\s]', ' ', text)

    # 4. Eliminación de espacios redundantes
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Aplicar el preprocesamiento al texto completo
print("Iniciando preprocesamiento con normalización de caracteres...")
df['texto_procesado'] = df['texto_completo'].apply(preprocess_text)
print("Preprocesamiento finalizado.")

# Mostrar un ejemplo del antes y después para validar
print("\n--- Ejemplo de Preprocesamiento ---")
print("ORIGINAL: ", df['texto_completo'].iloc[0])
print("PROCESADO:", df['texto_procesado'].iloc[0])

Iniciando preprocesamiento con normalización de caracteres...
Preprocesamiento finalizado.

--- Ejemplo de Preprocesamiento ---
ORIGINAL:  Percy Jackson & the Olympians: The Lightning Thief. A fantasy adventure that fuses Greek mythology to contemporary American places and values. Anyone around 15 (give or take a couple of years) will thrill to the visual spectacle
PROCESADO: percy jackson the olympians the lightning thief a fantasy adventure that fuses greek mythology to contemporary american places and values anyone around 15 give or take a couple of years will thrill to the visual spectacle


Generación de embeddings

In [5]:
# Celda 4: Generación de embeddings
# Cargamos el modelo pre-entrenado de libre elección
modelo_nombre = 'all-MiniLM-L6-v2'
print(f"Cargando el modelo: {modelo_nombre}...")
modelo = SentenceTransformer(modelo_nombre)

print("Generando embeddings para los documentos del corpus...")
print("(Esto puede tomar unos minutos dependiendo del tamaño del dataset)")

# Extraemos la columna de texto procesado como una lista de Python
textos_corpus = df['texto_procesado'].tolist()

# Generamos la matriz de representaciones vectoriales densas
# show_progress_bar=True activará una barra visual en Colab
doc_embeddings = modelo.encode(textos_corpus, show_progress_bar=True)

print("\n--- Resumen de Embeddings ---")
print(f"Cantidad de documentos procesados: {doc_embeddings.shape[0]}")
print(f"Dimensión de cada vector (embedding): {doc_embeddings.shape[1]}")

Cargando el modelo: all-MiniLM-L6-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generando embeddings para los documentos del corpus...
(Esto puede tomar unos minutos dependiendo del tamaño del dataset)


Batches:   0%|          | 0/8174 [00:00<?, ?it/s]


--- Resumen de Embeddings ---
Cantidad de documentos procesados: 261561
Dimensión de cada vector (embedding): 384


Procesamiento de consultas

In [6]:
# Celda 5: Requerimiento 3 - Procesamiento de consultas
from sklearn.metrics.pairwise import cosine_similarity

def procesar_consulta(query, modelo_embedding, corpus_embeddings):

    # 1. Recibir una consulta textual (
    query_procesada = preprocess_text(query)
    print(f"Paso 1 - Consulta procesada: '{query_procesada}'")

    # 2. Generar su embedding
    # encode() lo convierte en un vector denso. Lo ponemos en una lista [] para que el modelo lo entienda
    query_embedding = modelo_embedding.encode([query_procesada])
    print(f"Paso 2 - Embedding generado. Dimensiones: {query_embedding.shape}")

    # 3. Compararla con todos los documentos del corpus
    similitudes = cosine_similarity(query_embedding, corpus_embeddings)[0]
    print(f"Paso 3 - Comparación completada contra {len(similitudes)} documentos.")

    # Devolvemos el arreglo de puntajes matemáticos
    return similitudes

# Probamos que el procesamiento matemático funcione correctamente
print("--- Prueba del Requerimiento 3 ---")
puntajes_similitud = procesar_consulta("science fiction movie with advanced technology", modelo, doc_embeddings)

--- Prueba del Requerimiento 3 ---
Paso 1 - Consulta procesada: 'science fiction movie with advanced technology'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


 Recuperación

In [7]:
from IPython.display import display

def recuperar_documentos(similitudes, df_corpus, k=5):

    # 1. Obtener los índices ordenados de mayor a menor similitud (Top-k) [cite: 47]
    top_k_indices = np.argsort(similitudes)[::-1][:k]

    # 2. Construir la lista de resultados con los atributos obligatorios [cite: 47-49]
    resultados = []
    for rank, idx in enumerate(top_k_indices, start=1):
        # Validación de seguridad para no exceder el tamaño del dataset
        if idx < len(df_corpus):
            resultados.append({
                'Ranking': rank,                                        # Atributo Requerimiento 4 [cite: 49]
                'Identificador del documento': idx,                     # Atributo Requerimiento 4 [cite: 48]
                'Título película': df_corpus.iloc[idx]['movie_title'],  # Requisito de tabla [cite: 84]
                'Fragmento de texto': df_corpus.iloc[idx]['texto_completo'][:150] + "...", # Requisito de tabla [cite: 84]
                'Puntaje de similitud': round(similitudes[idx], 4)      # Atributo Requerimiento 4 [cite: 49]
            })

    # 3. Convertir la lista a un DataFrame de Pandas para la visualización en tabla
    return pd.DataFrame(resultados)

df_resultados_prueba = recuperar_documentos(puntajes_similitud, df, k=5)

# Mostrar la tabla formateada
display(df_resultados_prueba)

,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,9009,Surrogates,Surrogates. This movie does what science ficti...,0.6081
1,2,107497,Alien,Alien. An old-fashioned scary movie set in a h...,0.5764
2,3,101533,Aeon Flux,Aeon Flux. It's as if someone just swept up th...,0.5688
3,4,131842,Arrival,"Arrival. A high-stakes, hard sci-fi action fil...",0.5648
4,5,31674,Contact,Contact. One of the greatest science fiction f...,0.5636


Benchmark de consultas


In [8]:

# 1. Definimos el diccionario con las 8 consultas obligatorias [cite: 53-60]
consultas_benchmark = {
    "Q1": "science fiction movie with advanced technology",
    "Q2": "romantic story with emotional relationships",
    "Q3": "action movie with intense fight scenes",
    "Q4": "horror film that creates fear and suspense",
    "Q5": "visually impressive movie with weak storyline",
    "Q6": "emotionally moving performance by the lead actor",
    "Q7": "predictable plot but entertaining experience",
    "Q8": "movie praised by critics but unpopular with audiences"
}

resumen_general = []

print("="*90)
print("INICIANDO BENCHMARK DE CONSULTAS (Top-5 por consulta)")
print("="*90)

# 2. Ejecutamos el motor de búsqueda para cada consulta
for q_id, q_text in consultas_benchmark.items():
    print(f"\n\n{'='*40}")
    print(f"Buscando {q_id}: '{q_text}'")
    print(f"{'='*40}")

    # PASO A: Procesamos la consulta y obtenemos similitudes (Usando tu función del Requerimiento 3)
    # Nota: Esto imprimirá los mensajes de "Paso 1, 2 y 3" que definimos en esa celda
    similitudes_calculadas = procesar_consulta(q_text, modelo, doc_embeddings)

    # PASO B: Recuperamos los documentos en una tabla (Usando tu función del Requerimiento 4 con k=5)
    # Cada tabla incluirá: Ranking, ID, Título, Fragmento y Similitud [cite: 83-84]
    df_resultados = recuperar_documentos(similitudes_calculadas, df, k=5)

    # Mostramos la tabla Top-5 de esta consulta específica
    display(df_resultados)

    # PASO C: Guardamos el mejor resultado (Top-1) para la tabla resumen [cite: 86-87]
    top_1 = df_resultados.iloc[0]
    resumen_general.append({
        "Consulta": f"{q_id}",
        "Documento Top-1 (ID)": top_1['Identificador del documento'],
        "Título película": top_1['Título película'],
        "Similitud": top_1['Puntaje de similitud']
    })

# 3. Generamos y mostramos la tabla resumen general
print("\n\n" + "="*90)
print("TABLA RESUMEN GENERAL (Mejor resultado por consulta)")
print("="*90)
df_resumen = pd.DataFrame(resumen_general)
display(df_resumen)

INICIANDO BENCHMARK DE CONSULTAS (Top-5 por consulta)


Buscando Q1: 'science fiction movie with advanced technology'
Paso 1 - Consulta procesada: 'science fiction movie with advanced technology'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,9009,Surrogates,Surrogates. This movie does what science ficti...,0.6081
1,2,107497,Alien,Alien. An old-fashioned scary movie set in a h...,0.5764
2,3,101533,Aeon Flux,Aeon Flux. It's as if someone just swept up th...,0.5688
3,4,131842,Arrival,"Arrival. A high-stakes, hard sci-fi action fil...",0.5648
4,5,31674,Contact,Contact. One of the greatest science fiction f...,0.5636




Buscando Q2: 'romantic story with emotional relationships'
Paso 1 - Consulta procesada: 'romantic story with emotional relationships'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,154119,Beautiful Boy,Beautiful Boy. A story about a relationship th...,0.6878
1,2,33205,At First Sight,At First Sight. A romantic drama with insights...,0.6812
2,3,225003,Circumstance,Circumstance. An emotionally dense lesbian lov...,0.6650
3,4,119809,Amour,Amour. A poignant tale of undying love!...,0.6491
4,5,154775,Beautiful Something,Beautiful Something. A compassionate romantic ...,0.6312




Buscando Q3: 'action movie with intense fight scenes'
Paso 1 - Consulta procesada: 'action movie with intense fight scenes'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


...

,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,30097,Twister,Twister. Even though the plot is more predicta...,0.6816
1,2,93379,A Slipping-Down Life,A Slipping-Down Life. The predictable plot lur...,0.6765
2,3,43183,Insomnia,Insomnia. ...if there is anything that spoils ...,0.6717
3,4,10798,Orphan,"Orphan. Predictable and been-there, seen-that,...",0.6706
4,5,43318,Trapped,"Trapped. Never engaging, utterly predictable a...",0.6646




Buscando Q8: 'movie praised by critics but unpopular with audiences'
Paso 1 - Consulta procesada: 'movie praised by critics but unpopular with audiences'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,28116,The Kid,The Kid. Critics at the time praised the film ...,0.6506
1,2,185439,Boy,"Boy. It is far from a bad film; in fact, it's ...",0.6437
2,3,135528,ATL,"ATL. A bland, lackluster film that thanks to b...",0.6365
3,4,111564,Almost Famous,Almost Famous. A good but not great little mov...,0.6358
4,5,9039,The Promotion,The Promotion. I predict that when this film i...,0.6268




TABLA RESUMEN GENERAL (Mejor resultado por consulta)


,Consulta,Documento Top-1 (ID),Título película,Similitud
0,Q1,9009,Surrogates,0.6081
1,Q2,154119,Beautiful Boy,0.6878
2,Q3,69252,Warrior,0.7011
3,Q4,18330,Fright Night,0.7878
4,Q5,20515,M,0.6893
5,Q6,159013,Being There,0.6636
6,Q7,30097,Twister,0.6816
7,Q8,28116,The Kid,0.6506


## Desafio de Excelencia

In [14]:
# Celda 8: Desafío de Excelencia - Comparación de Modelos de Embeddings [cite: 101-107]
# Solo importamos las herramientas nuevas que no se han cargado en celdas anteriores
import gensim.downloader as api
from tqdm import tqdm

print("="*90)
print("DESAFÍO DE EXCELENCIA: COMPARACIÓN DE MODELOS (MiniLM vs. Word2Vec)")
print("="*90)

# 1. Cargar el segundo modelo (Word2Vec)
print("\nDescargando y cargando 'word2vec-google-news-300' (~1.6 GB)...")
modelo_w2v = api.load("word2vec-google-news-300")
print("Modelo Word2Vec cargado exitosamente.")

# 2. Función para generar el embedding de un documento promediando sus palabras
def vector_documento_w2v(texto, modelo):
    palabras = str(texto).split()
    palabras_validas = [p for p in palabras if p in modelo]
    if palabras_validas:
        return np.mean(modelo[palabras_validas], axis=0)
    else:
        return np.zeros(modelo.vector_size)

# 3. Generar embeddings del corpus usando Word2Vec
print("\nGenerando matriz de embeddings con Word2Vec (Promediado de palabras)...")
# Usamos tqdm con pandas para ver la barra de progreso
tqdm.pandas(desc="Progreso Word2Vec")
w2v_embeddings_list = df['texto_procesado'].progress_apply(lambda x: vector_documento_w2v(x, modelo_w2v))
w2v_corpus_embeddings = np.vstack(w2v_embeddings_list.values)

print(f"Matriz Word2Vec generada. Dimensiones: {w2v_corpus_embeddings.shape}")

# 4. Función de búsqueda adaptada para Word2Vec
def buscar_w2v(query, modelo_w2v, corpus_embeddings_w2v, df_corpus, k=5):
    query_procesada = preprocess_text(query)
    query_embedding = vector_documento_w2v(query_procesada, modelo_w2v)

    # Calcular similitud (hacemos reshape porque es un solo vector)
    similitudes = cosine_similarity(query_embedding.reshape(1, -1), corpus_embeddings_w2v)[0]
    top_k_indices = np.argsort(similitudes)[::-1][:k]

    resultados = []
    for rank, idx in enumerate(top_k_indices, start=1):
        if idx < len(df_corpus):
            resultados.append({
                'Ranking': rank,
                'ID documento': idx,
                'Título película': df_corpus.iloc[idx]['movie_title'],
                'Fragmento de texto': df_corpus.iloc[idx]['texto_completo'][:150] + "...",
                'Similitud': round(similitudes[idx], 4)
            })
    return pd.DataFrame(resultados)

# 5. Ejecución de la comparación directa con una consulta del Benchmark
consulta_comparativa = "science fiction movie with advanced technology" # Q1

print("\n\n" + "="*90)
print(f"COMPARATIVA PARA LA CONSULTA: '{consulta_comparativa}'")
print("="*90)

print("\n--- RESULTADOS MODELO 1: all-MiniLM-L6-v2 (Contexto Semántico) ---")
# Usamos las variables que ya tenías en la memoria RAM de las celdas anteriores
df_minilm = recuperar_documentos(procesar_consulta(consulta_comparativa, modelo, doc_embeddings), df, k=5)
display(df_minilm)

print("\n--- RESULTADOS MODELO 2: word2vec-google-news-300 (Promedio de Palabras) ---")
df_word2vec = buscar_w2v(consulta_comparativa, modelo_w2v, w2v_corpus_embeddings, df, k=5)
display(df_word2vec)

# Breve conclusión analítica impresa en el notebook
print("\n" + "-"*90)
print("ANÁLISIS DE LA COMPARACIÓN:")
print("Se observa que 'all-MiniLM-L6-v2' logra capturar mejor la intención general de la oración completa.")
print("Por su parte, al promediar los vectores de 'word2vec', el modelo pierde el orden y la estructura")
print("gramatical, dándole un peso equitativo a cada palabra individual, lo que en tareas de recuperación")
print("documental extensa suele generar más falsos positivos en la similitud coseno.")
print("-"*90)

DESAFÍO DE EXCELENCIA: COMPARACIÓN DE MODELOS (MiniLM vs. Word2Vec)

Descargando y cargando 'word2vec-google-news-300' (~1.6 GB)...
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Modelo Word2Vec cargado exitosamente.

Generando matriz de embeddings con Word2Vec (Promediado de palabras)...


Progreso Word2Vec: 100%|██████████| 261561/261561 [00:39<00:00, 6608.82it/s]


Matriz Word2Vec generada. Dimensiones: (261561, 300)


COMPARATIVA PARA LA CONSULTA: 'science fiction movie with advanced technology'

--- RESULTADOS MODELO 1: all-MiniLM-L6-v2 (Contexto Semántico) ---
Paso 1 - Consulta procesada: 'science fiction movie with advanced technology'
Paso 2 - Embedding generado. Dimensiones: (1, 384)
Paso 3 - Comparación completada contra 261561 documentos.


,Ranking,Identificador del documento,Título película,Fragmento de texto,Puntaje de similitud
0,1,9009,Surrogates,Surrogates. This movie does what science ficti...,0.6081
1,2,107497,Alien,Alien. An old-fashioned scary movie set in a h...,0.5764
2,3,101533,Aeon Flux,Aeon Flux. It's as if someone just swept up th...,0.5688
3,4,131842,Arrival,"Arrival. A high-stakes, hard sci-fi action fil...",0.5648
4,5,31674,Contact,Contact. One of the greatest science fiction f...,0.5636



--- RESULTADOS MODELO 2: word2vec-google-news-300 (Promedio de Palabras) ---


,Ranking,ID documento,Título película,Fragmento de texto,Similitud
0,1,128169,Anya,Anya. Science fiction with an emphasis on the ...,0.7692
1,2,42206,The Time Machine,The Time Machine. A very average science ficti...,0.7502
2,3,24391,The Thing from Another World,The Thing from Another World. Lively science f...,0.7356
3,4,23002,RoboCop,RoboCop. Prime science fiction....,0.7355
4,5,124681,Annihilation,Annihilation. Everything a modern science fict...,0.7342



------------------------------------------------------------------------------------------
ANÁLISIS DE LA COMPARACIÓN:
Se observa que 'all-MiniLM-L6-v2' logra capturar mejor la intención general de la oración completa.
Por su parte, al promediar los vectores de 'word2vec', el modelo pierde el orden y la estructura
gramatical, dándole un peso equitativo a cada palabra individual, lo que en tareas de recuperación
documental extensa suele generar más falsos positivos en la similitud coseno.
------------------------------------------------------------------------------------------
